<a href="https://colab.research.google.com/github/KgadiKatse/Insurance-dataset/blob/main/Health_Insurance_Claims_Data_for_Fraud_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Healthcare FWA intelligence — provider behavior, utilization, coding, networks and SHAP

**Reproducible extension** to the 20,100-row Kaggle synthetic claims analysis.
The analysis uses a **chronological** train/validation/test split, excludes all claims
carrying repeated Claim_ID values, keeps test labels out of feature building and
threshold selection, and reports model results only on the untouched test period.

**Interpretation:** Diagnosis–procedure *rarity is not clinical incompatibility*.
Provider-level signals are illustrative because Hospital_ID is not a stable practice
identity in the synthetic generator. Scores and links are leads for review only.

Procedures are five-digit US-style CPT codes, not Botswana NAPPI/NRPL or ICD-10-PCS.

*Clinical necessity should be assessed with licensed, versioned coding and payer rules.*

In [ ]:
# ── ENVIRONMENT SET UP────────────────────────────────────────────────────────

# ── Core libraries ───────────────────────────────────────────────────────────
from pathlib import Path
from collections import Counter, defaultdict
import zipfile, json, warnings, math
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score,
                             precision_score, recall_score, precision_recall_curve)
from xgboost import XGBClassifier
import shap

# ── Reproducibility ──────────────────────────────────────────────────────────
# Fixing the seed means everyone who runs this notebook sees the same

SEED = 42
ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
ARCHIVE = ROOT / 'archive.zip'
if not ARCHIVE.exists(): ARCHIVE = Path('/mnt/data/archive.zip')
OUT = ROOT / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tejalaveti2306/health-insurance-claims-data-for-fraud-detection")

print("Path to dataset files:", path)

## 1. Exploratory Data Analysis